In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

In [ ]:

# load files
sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")
sell_prices = pd.read_csv("sell_prices.csv")

# keep id columns separately
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

# wide -> long
sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# join calendar info
df = sales_long.merge(calendar, on="d", how="left")

# join prices using Walmart week key
df = df.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# date handling
df["date"] = pd.to_datetime(df["date"])

# reduce memory a bit
for col in ["sales", "sell_price"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

# keep categorical columns as category dtype
cat_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

# sort before feature engineering
df = df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)


(58327370, 22)
                            id      item_id  dept_id cat_id store_id state_id  \
0  FOODS_1_001_CA_1_validation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
1  FOODS_1_001_CA_1_validation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
2  FOODS_1_001_CA_1_validation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
3  FOODS_1_001_CA_1_validation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
4  FOODS_1_001_CA_1_validation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   

     d  sales       date  wm_yr_wk  ... month  year  event_name_1  \
0  d_1    3.0 2011-01-29     11101  ...     1  2011           NaN   
1  d_2    0.0 2011-01-30     11101  ...     1  2011           NaN   
2  d_3    0.0 2011-01-31     11101  ...     1  2011           NaN   
3  d_4    1.0 2011-02-01     11101  ...     2  2011           NaN   
4  d_5    4.0 2011-02-02     11101  ...     2  2011           NaN   

   event_type_1 event_name_2 event_type_2 snap_CA snap_TX  snap_WI  sell_price  
0 

In [4]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price'],
      dtype='str')